In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END

def mock_llm(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "hello world"}]}

graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)
graph = graph.compile()

graph.invoke({"messages": [{"role": "user", "content": "hi!"}]})

{'messages': [HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='093fc7aa-3921-42fb-b583-ccbaca2ae349'),
  AIMessage(content='hello world', additional_kwargs={}, response_metadata={}, id='f3602cca-e851-4036-8200-2e22f23b3bcc', tool_calls=[], invalid_tool_calls=[])]}

In [59]:
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3.5",
    validate_model_on_init=True,
    temperature=0.8,
    num_predict=4096,

    # other params ...
)

class SearchQuery(BaseModel):
    search_query: str = Field(..., description="Query that is optimized web search.")
    justification: str = Field(
        ..., description="Why this query is relevant to the user's request."
    )

# Creating structured output
structured_llm = llm.with_structured_output(SearchQuery)

output = llm.invoke("What is the role of UvsX in recombination?")


ValidationError: 1 validation error for ChatOllama
  Value error, Model `gemma4` not found in Ollama. Please pull the model (using `ollama pull gemma4`) or specify a valid model name. Available local models: qwen3.5:latest [type=value_error, input_value={'model': 'gemma4', 'vali....8, 'num_predict': 4096}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [55]:
# Define a tool
def multiply(a: int, b: int) -> int:
    return a * b

# Augment the LLM with tools
llm_with_tools = llm.bind_tools([multiply])

# Invoke the LLM with input that triggers the tool call
msg = llm_with_tools.invoke("What is 2 times 3?")

# Get the tool call
msg.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': '1fb4edbf-b900-497f-b740-d50d4b85121b',
  'type': 'tool_call'}]

In [60]:
msg

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-20T13:35:04.600044Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3880648250, 'load_duration': 77283083, 'prompt_eval_count': 276, 'prompt_eval_duration': 816187125, 'eval_count': 80, 'eval_duration': 2968616668, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019dab19-fe6e-75c2-b613-1c9ca3772e76-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': '1fb4edbf-b900-497f-b740-d50d4b85121b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 276, 'output_tokens': 80, 'total_tokens': 356})